In [2]:
!pip install -q sentence-transformers supabase google-genai beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.7 MB/s eta 0:00:00


In [3]:
from sentence_transformers import SentenceTransformer
from supabase import create_client
from google import genai
from google.colab import userdata

model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, matches spring_boot_docs.embedding column

SUPABASE_URL = "https://vpwrpyzipigtsgwotnxr.supabase.co"
supabase = create_client(SUPABASE_URL, userdata.get('SUPABASE_SERVICE_KEY'))

gemini_client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
LLM_MODEL = "gemini-flash-latest"  # alias — auto-tracks Google's current recommended flash model

print("Setup complete: model, supabase, gemini_client ready")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete: model, supabase, gemini_client ready


In [4]:
import requests
from bs4 import BeautifulSoup

HEADING_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6"}
TEXT_TAGS = {"p", "li", "pre", "td", "th", "blockquote"}
CHUNK_SIZE = 800


def scrape_page(url):
    """Scrapes one doc page into text blocks, tagging each with its most recent heading."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    content_root = soup.find("article") or soup.find("main") or soup.body

    blocks = []
    current_section = "Introduction"
    for el in content_root.find_all(HEADING_TAGS | TEXT_TAGS):
        text = el.get_text(separator=" ", strip=True)
        if not text:
            continue
        if el.name in HEADING_TAGS:
            current_section = text
            continue
        if len(text) < 3:
            continue
        blocks.append({"text": text, "section_title": current_section, "source_url": url})
    return blocks


def chunk_blocks(blocks, chunk_size=CHUNK_SIZE):
    """Groups blocks into chunks, flushing whenever the section heading changes."""
    chunks = []
    buffer, buffer_section, buffer_url = "", None, None

    def flush():
        if buffer.strip():
            chunks.append({"content": buffer.strip(), "section_title": buffer_section, "source_url": buffer_url})

    for block in blocks:
        if buffer_section is not None and block["section_title"] != buffer_section:
            flush()
            buffer = ""
        if not buffer:
            buffer_section, buffer_url = block["section_title"], block["source_url"]
        candidate = (buffer + " " + block["text"]).strip() if buffer else block["text"]
        if len(candidate) > chunk_size and buffer:
            flush()
            buffer = block["text"]
        else:
            buffer = candidate
    flush()
    return chunks


print("scrape_page and chunk_blocks defined")

scrape_page and chunk_blocks defined


In [5]:
def ingest_pages(urls):
    all_chunks = []
    for url in urls:
        blocks = scrape_page(url)
        page_chunks = chunk_blocks(blocks)
        all_chunks.extend(page_chunks)
        print(f"{url} -> {len(blocks)} blocks, {len(page_chunks)} chunks")

    existing = supabase.table("spring_boot_docs").select("content").execute()
    existing_content = {row["content"] for row in existing.data}

    new_chunks = [c for c in all_chunks if c["content"] not in existing_content]
    skipped = len(all_chunks) - len(new_chunks)
    if skipped:
        print(f"Skipping {skipped} chunks already in the database")

    if not new_chunks:
        print("Nothing new to insert.")
        return

    texts = [c["content"] for c in new_chunks]
    embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

    rows = [
        {
            "content": chunk["content"],
            "section_title": chunk["section_title"],
            "source_url": chunk["source_url"],
            "embedding": emb.tolist(),
        }
        for chunk, emb in zip(new_chunks, embeddings)
    ]

    batch_size = 10
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        supabase.table("spring_boot_docs").insert(batch).execute()
        print(f"Inserted {i + len(batch)}/{len(rows)}")

    print(f"Done. Inserted {len(rows)} new chunks.")


print("ingest_pages defined — call with a list of doc URLs to add more pages safely")

ingest_pages defined — call with a list of doc URLs to add more pages safely


In [6]:
def retrieve_chunks(question: str, top_k: int = 5):
    query_embedding = model.encode(question, normalize_embeddings=True).tolist()
    response = supabase.rpc(
        "match_documents",
        {"query_embedding": query_embedding, "match_count": top_k}
    ).execute()
    return response.data


print("retrieve_chunks defined")


retrieve_chunks defined


In [7]:
def build_context(chunks):
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(f"[Source {i}: {c['section_title']}]\n{c['content']}")
    return "\n\n".join(parts)


def ask_rag(question: str, top_k: int = 5):
    chunks = retrieve_chunks(question, top_k=top_k)
    context = build_context(chunks)

    prompt = f"""You are a helpful assistant answering questions about Spring Boot based ONLY on the provided documentation excerpts below. If the excerpts don't contain enough information to answer, say so clearly rather than guessing.

Documentation excerpts:
{context}

Question: {question}

Answer concisely, and reference which source section(s) you used."""

    response = gemini_client.models.generate_content(model=LLM_MODEL, contents=prompt)

    print(f"Question: {question}\n")
    print(f"Answer:\n{response.text}\n")
    print("Sources used:")
    for c in chunks:
        print(f"  - {c['section_title']} ({c['source_url']}) [similarity={c['similarity']:.3f}]")

    return response.text, chunks


print("ask_rag defined — ready to use")


ask_rag defined — ready to use


In [8]:
ingest_pages([
    "https://docs.spring.io/spring-boot/reference/data/sql.html",
    "https://docs.spring.io/spring-boot/reference/web/servlet.html",
    "https://docs.spring.io/spring-boot/reference/data/nosql.html",
    "https://docs.spring.io/spring-boot/reference/web/spring-security.html",
])

answer_sql, sources_sql = ask_rag("How does Spring Boot configure connection pooling?")
answer_servlet, sources_servlet = ask_rag("How does Spring Boot handle servlet filters?")
answer_nosql, sources_nosql = ask_rag("How do I connect Spring Boot to a MongoDB database?")
answer_security, sources_security = ask_rag("What does Spring Boot auto-configure by default when Spring Security is on the classpath?")

https://docs.spring.io/spring-boot/reference/data/sql.html -> 265 blocks, 67 chunks
https://docs.spring.io/spring-boot/reference/web/servlet.html -> 349 blocks, 96 chunks
https://docs.spring.io/spring-boot/reference/data/nosql.html -> 388 blocks, 77 chunks
https://docs.spring.io/spring-boot/reference/web/spring-security.html -> 42 blocks, 14 chunks
Skipping 254 chunks already in the database
Nothing new to insert.
Question: How does Spring Boot configure connection pooling?

Answer:
Spring Boot configures connection pooling through the following mechanisms:

1. **Auto-Configuration Algorithm:** Spring Boot checks for pooled connection implementations in a specific order: 
   * HikariCP (preferred/default if available, included automatically via `spring-boot-starter-jdbc` or `spring-boot-starter-data-jpa`)
   * Tomcat pooling DataSource
   * Commons DBCP2
   * Oracle UCP  
   *(Source 1, Source 2)*

2. **Driver Verification:** Before creating a pooling `DataSource`, Spring Boot verifies